# OULAD Early-Risk Classification

Primary cutoff: 28 days. Secondary experiment: 56 days. Target: `Fail`/`Withdrawn` = 1, `Pass`/`Distinction` = 0.

The split is grouped by `id_student` so the same learner cannot appear in train and test.

In [ ]:
import os, sys, zipfile, random, subprocess, warnings
warnings.filterwarnings('ignore')

REPO_DIR='/content/Summative_Assignment-Model_Training_and_Evaluation'
REPO_URL='https://github.com/Abejirin-King/Summative_Assignment-Model_Training_and_Evaluation.git'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git','clone','-q',REPO_URL,REPO_DIR],check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0,REPO_DIR)

try:
    import tensorflow as tf
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','tensorflow>=2.15'],check=True)
    import tensorflow as tf

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, roc_curve, precision_recall_curve
from src.feature_engineering import build_dataset
from src.models import make_preprocessor, traditional_models, make_pipeline
from src.deep_learning import make_tf_datasets, build_mlp, train_mlp

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
OUT_DIR=os.path.join(REPO_DIR,'outputs'); FIG_DIR=os.path.join(OUT_DIR,'figures'); LOG_DIR=os.path.join(OUT_DIR,'tensorboard')
os.makedirs(FIG_DIR,exist_ok=True); os.makedirs(LOG_DIR,exist_ok=True)
print('Repository:',os.getcwd()); print('TensorFlow:',tf.__version__)

In [ ]:
ZIP_PATH='/content/anonymisedData.zip'; DATA_DIR='/content/data'
if not os.path.exists(ZIP_PATH): raise FileNotFoundError('Upload anonymisedData.zip to Colab, then rerun this cell.')
if not os.path.exists(os.path.join(DATA_DIR,'studentInfo.csv')):
    os.makedirs(DATA_DIR,exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(DATA_DIR)
required={'studentInfo.csv','studentAssessment.csv','assessments.csv','studentRegistration.csv','studentVle.csv','vle.csv','courses.csv'}
missing=required-set(os.listdir(DATA_DIR))
if missing: raise FileNotFoundError(f'Missing OULAD files: {sorted(missing)}')
df=build_dataset(DATA_DIR,28)
print('Raw studentInfo rows:',len(pd.read_csv(os.path.join(DATA_DIR,'studentInfo.csv'))))
print('28-day dataset:',df.shape)
print(df['at_risk'].value_counts(normalize=True))

In [5]:
def split(df):
    X=df.drop(columns=['at_risk','id_student']); y=df.at_risk.astype(int); g=df.id_student
    a,b=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(X,y,g))
    Xtr,Xte=X.iloc[a],X.iloc[b]; ytr,yte=y.iloc[a],y.iloc[b]; gt=g.iloc[a]
    c,d=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=43).split(Xtr,ytr,gt))
    return Xtr.iloc[c],Xtr.iloc[d],Xte,ytr.iloc[c],ytr.iloc[d],yte
Xtr,Xval,Xtest,ytr,yval,ytest=split(df)
for name,model in traditional_models().items():
    pipe=make_pipeline(Xtr,model); pipe.fit(Xtr,ytr); p=pipe.predict_proba(Xtest)[:,1]
    print(name, 'AUC=',round(roc_auc_score(ytest,p),4),'F1=',round(f1_score(ytest,p>=.5),4))


LogisticRegression AUC= 0.8307 F1= 0.7451
RandomForest AUC= 0.837 F1= 0.7424
HistGradientBoosting AUC= 0.8425 F1= 0.7586


## Clean experiment pipeline
The cells below replace the earlier scratch-style evaluation. They provide grouped splitting, validation-only threshold selection, 10 recorded experiments, TensorBoard scalar logs, required plots, and model export.

In [ ]:
def grouped_split(data, test_size=.20, val_size=.20, random_state=42):
    X=data.drop(columns=['at_risk','id_student'])
    y=data['at_risk'].astype(int)
    groups=data['id_student']
    outer=GroupShuffleSplit(n_splits=1,test_size=test_size,random_state=random_state)
    trv,te=next(outer.split(X,y,groups))
    Xtv,ytv,gtv=X.iloc[trv],y.iloc[trv],groups.iloc[trv]
    inner=GroupShuffleSplit(n_splits=1,test_size=val_size,random_state=random_state+1)
    tr,va=next(inner.split(Xtv,ytv,gtv))
    return Xtv.iloc[tr].copy(),Xtv.iloc[va].copy(),X.iloc[te].copy(),ytv.iloc[tr].copy(),ytv.iloc[va].copy(),y.iloc[te].copy(),gtv.iloc[tr],gtv.iloc[va],groups.iloc[te]

Xtr,Xval,Xtest,ytr,yval,ytest,gtr,gval,gtest=grouped_split(df)
print('Train/val/test:',len(Xtr),len(Xval),len(Xtest))
print('Unique students:',gtr.nunique(),gval.nunique(),gtest.nunique())
print('Student overlap:',len(set(gtr)&set(gval)),len(set(gtr)&set(gtest)),len(set(gval)&set(gtest)))

In [ ]:
results=[]

def evaluate(y_true,p,threshold=.5):
    pred=(np.asarray(p)>=threshold).astype(int)
    return {'accuracy':accuracy_score(y_true,pred),'precision':precision_score(y_true,pred,zero_division=0),'recall':recall_score(y_true,pred,zero_division=0),'f1':f1_score(y_true,pred,zero_division=0),'roc_auc':roc_auc_score(y_true,p),'pr_auc':average_precision_score(y_true,p),'threshold':float(threshold)}

def best_threshold(y_true,p):
    return max(((float(t),f1_score(y_true,p>=t,zero_division=0)) for t in np.arange(.20,.71,.05)),key=lambda x:x[1])[0]

def record(name,family,split_name,metrics,settings):
    results.append({'experiment':name,'family':family,'split':split_name,**metrics,'settings':settings})
    writer=tf.summary.create_file_writer(os.path.join(LOG_DIR,name))
    with writer.as_default():
        for k,v in metrics.items():
            if np.isscalar(v) and np.isfinite(v): tf.summary.scalar(k,float(v),step=0)
    writer.flush(); writer.close()

traditional_pipes={}; traditional_predictions={}
for name,model in traditional_models().items():
    pipe=make_pipeline(Xtr,model); pipe.fit(Xtr,ytr)
    pval=pipe.predict_proba(Xval)[:,1]; ptest=pipe.predict_proba(Xtest)[:,1]
    record(name,'Traditional','28-day',evaluate(ytest,ptest,.5),'full features; threshold=0.50')
    traditional_pipes[name]=pipe; traditional_predictions[name]=ptest

hgb=traditional_pipes['HistGradientBoosting']
hgb_val_p=hgb.predict_proba(Xval)[:,1]; hgb_test_p=hgb.predict_proba(Xtest)[:,1]
hgb_threshold=best_threshold(yval,hgb_val_p)
record('HGB_threshold_tuned','Traditional','28-day',evaluate(ytest,hgb_test_p,hgb_threshold),f'full features; validation threshold={hgb_threshold:.2f}')
print('HGB validation threshold:',hgb_threshold)

In [ ]:
def feature_groups(data):
    cols=[c for c in data.columns if c not in {'at_risk','id_student'}]
    static_names={'gender','region','highest_education','imd_band','age_band','num_of_prev_attempts','studied_credits','disability','code_module','code_presentation','registration_day'}
    static=[c for c in cols if c in static_names]
    engagement=[c for c in cols if c.startswith(('early_total_','early_active_','early_resource_','early_max_','early_mean_','early_click_','week_','engagement_','active_day_','clicks_per_'))]
    return static,engagement

static,engagement=feature_groups(df)
for name,cols in [('Ablation_static_only',static),('Ablation_static_plus_engagement',static+engagement)]:
    pipe=make_pipeline(Xtr[cols],traditional_models()['HistGradientBoosting']); pipe.fit(Xtr[cols],ytr)
    record(name,'Ablation','28-day',evaluate(ytest,pipe.predict_proba(Xtest[cols])[:,1],.5),f'{len(cols)} features')
print('Feature counts:',len(static),len(engagement))

### Deep learning experiment grid
Three architectures are compared with identical preprocessing and grouped splits. Each run uses tf.data, early stopping and learning-rate reduction.

In [ ]:
prep_dl=make_preprocessor(Xtr)
A=prep_dl.fit_transform(Xtr).astype('float32'); B=prep_dl.transform(Xval).astype('float32'); C=prep_dl.transform(Xtest).astype('float32')
train_ds,val_ds=make_tf_datasets(A,ytr.to_numpy(),B,yval.to_numpy(),128)

configs=[((128,64,32),.30,1e-3,'MLP_128_64_32'),((128,64),.20,1e-3,'MLP_128_64'),((256,128,64),.35,5e-4,'MLP_256_128_64')]
dl_runs=[]
for units,dropout,lr,name in configs:
    tf.keras.backend.clear_session(); tf.random.set_seed(SEED)
    model=build_mlp(A.shape[1],units,dropout,lr)
    history=train_mlp(model,train_ds,val_ds,80)
    logdir=os.path.join(LOG_DIR,name); writer=tf.summary.create_file_writer(logdir)
    with writer.as_default():
        for epoch in range(len(history.history['val_roc_auc'])):
            for key in ['loss','val_loss','roc_auc','val_roc_auc']:
                tf.summary.scalar(key,float(history.history[key][epoch]),step=epoch)
    writer.flush(); writer.close()
    pval=model.predict(B,batch_size=256,verbose=0).ravel(); ptest=model.predict(C,batch_size=256,verbose=0).ravel()
    threshold=best_threshold(yval,pval); metrics=evaluate(ytest,ptest,threshold)
    metrics['best_val_roc_auc']=float(max(history.history['val_roc_auc'])); metrics['best_epoch']=int(np.argmax(history.history['val_roc_auc'])+1)
    record(name,'Deep Learning','28-day',metrics,f'units={units}; dropout={dropout}; lr={lr}; batch=128')
    dl_runs.append({'name':name,'model':model,'history':history.history,'ptest':ptest,'metrics':metrics})
best_dl_run=max(dl_runs,key=lambda r:r['metrics']['roc_auc'])
best_dl=best_dl_run['model']; best_dl_history=best_dl_run['history']; best_dl_test_p=best_dl_run['ptest']
print('Best DL:',best_dl_run['name'],round(best_dl_run['metrics']['roc_auc'],4))

In [ ]:
df56=build_dataset(DATA_DIR,56)
X56tr,X56val,X56test,y56tr,y56val,y56test,_,_,_=grouped_split(df56)
hgb56=make_pipeline(X56tr,traditional_models()['HistGradientBoosting']); hgb56.fit(X56tr,y56tr)
p56=hgb56.predict_proba(X56test)[:,1]
record('HGB_56_day','Traditional','56-day',evaluate(y56test,p56,.5),'full 56-day features; threshold=0.50')
results_df=pd.DataFrame(results)
display_cols=['experiment','family','split','accuracy','precision','recall','f1','roc_auc','pr_auc','threshold']
display(results_df[display_cols].sort_values('roc_auc',ascending=False).round(4))
results_df.to_csv(os.path.join(OUT_DIR,'experiment_results.csv'),index=False)

## Required evaluation plots
The following cells generate confusion matrices, ROC curves, precision-recall curves, and learning curves for the final 28-day comparison.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,(title,p,thr) in zip(axes,[('HGB',hgb_test_p,hgb_threshold),('Best MLP',best_dl_test_p,best_dl_run['metrics']['threshold'])]):
    cm=confusion_matrix(ytest,p>=thr); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',cbar=False,ax=ax)
    ax.set_title(f'{title} confusion matrix'); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,'confusion_matrices.png'),dpi=200,bbox_inches='tight'); plt.show()

plt.figure(figsize=(7,5))
for label,p in [('HGB',hgb_test_p),('Best MLP',best_dl_test_p)]:
    fpr,tpr,_=roc_curve(ytest,p); plt.plot(fpr,tpr,label=f'{label} AUC={roc_auc_score(ytest,p):.3f}')
plt.plot([0,1],[0,1],'k--'); plt.xlabel('False positive rate'); plt.ylabel('True positive rate'); plt.title('ROC curves'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,'roc_curves.png'),dpi=200,bbox_inches='tight'); plt.show()

plt.figure(figsize=(7,5))
for label,p in [('HGB',hgb_test_p),('Best MLP',best_dl_test_p)]:
    precision,recall,_=precision_recall_curve(ytest,p); plt.plot(recall,precision,label=label)
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-recall curves'); plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR,'precision_recall_curves.png'),dpi=200,bbox_inches='tight'); plt.show()

In [ ]:
h=best_dl_history
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(h['loss'],label='train loss'); axes[0].plot(h['val_loss'],label='validation loss'); axes[0].set_title('MLP loss learning curve'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(h['roc_auc'],label='train ROC-AUC'); axes[1].plot(h['val_roc_auc'],label='validation ROC-AUC'); axes[1].set_title('MLP ROC-AUC learning curve'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,'learning_curves.png'),dpi=200,bbox_inches='tight'); plt.show()
print('Best validation epoch:',best_dl_run['metrics']['best_epoch'])
print('Best validation ROC-AUC:',round(best_dl_run['metrics']['best_val_roc_auc'],4))

## Model export
The selected traditional and deep-learning models are saved as required. The traditional model is chosen from the 28-day traditional experiments. Raw data and binaries remain outside GitHub.

In [ ]:
traditional_28=[r for r in results if r['family']=='Traditional' and r['split']=='28-day']
best_traditional_name=max(traditional_28,key=lambda r:r['roc_auc'])['experiment']
best_traditional=traditional_pipes['HistGradientBoosting'] if best_traditional_name.startswith('HGB') else traditional_pipes[best_traditional_name]
MODEL_DIR=os.path.join(OUT_DIR,'models'); os.makedirs(MODEL_DIR,exist_ok=True)
joblib.dump(best_traditional,os.path.join(MODEL_DIR,'best_traditional_model.joblib'))
best_dl.save(os.path.join(MODEL_DIR,'best_deep_learning_model.keras'))
joblib.dump(prep_dl,os.path.join(MODEL_DIR,'deep_learning_preprocessor.joblib'))
print('Best traditional:',best_traditional_name)
print('Best DL:',best_dl_run['name'])
print('Saved:',os.listdir(MODEL_DIR))

## Final interpretation
Use experiment_results.csv and the generated figures as the authoritative results for the report. Discuss algorithm differences, threshold trade-offs, the contribution of early engagement, learning-curve evidence, and the 28-day versus 56-day timeliness trade-off. Do not copy stale hard-coded results from previous runs.